**Navigation** : [Index](../../README.md) | [<< Previous](01-1-OpenAI-DALL-E-3.ipynb) | [Next >>](01-3-Basic-Image-Operations.ipynb)

# GPT-5 Multimodal - Image Analysis and Generation

**Module :** 01-Images-Foundation
**Level :** 🟢 Beginner
**Technologies :** GPT-5, OpenRouter API, Vision AI
**Estimated duration :** 30 minutes

## 🎯 Learning Objectives

- [ ] Configure GPT-5 via OpenRouter for image analysis
- [ ] Master multimodal text + image conversations
- [ ] Analyze and describe images with precision
- [ ] Create optimized prompts for visual analysis
- [ ] Integrate GPT-5 into pedagogical use cases

## 📚 Prerequisites

- Environment Setup (module 00) completed
- OpenRouter API key configured
- Basic knowledge of multimodal AI

## ⚡ GPT-5 Capabilities

- **Advanced vision** : Detailed image analysis
- **Multimodal** : Simultaneous text + image conversation
- **Extended context** : Up to 400K tokens [OpenAI 2025]
- **Reasoning** : Complex and deductive analysis
- **Educational** : Perfect for teaching

In [1]:
# Parametres Papermill - JAMAIS modifier ce commentaire

# Configuration conversation
notebook_mode = "interactive"        # "interactive" ou "batch"
skip_widgets = False               # True pour mode batch MCP
debug_level = "INFO"               

# Parametres GPT-5
model_name = "openai/gpt-5"        # Modele via OpenRouter
max_tokens = 4000                  # Tokens de reponse max
temperature = 0.7                  # Creativite (0.0-1.0)
top_p = 0.9                        # Diversite sampling

# Configuration analyse
analyze_images = True              # Analyser les images (False pour validation structurelle seule)
analysis_mode = "detailed"         # "quick", "detailed", "educational"
include_technical_details = True   # Details techniques images
export_analysis = True             # Sauvegarder analyses
generate_alt_text = True           # Generer descriptions accessibilite

# Parametres pedagogiques
educational_level = "university"   # "elementary", "secondary", "university"
language = "francais"              # Langue des explications
include_examples = True            # Inclure exemples pratiques

In [2]:
# Parameters
BATCH_MODE = "true"


## Environment setup

Let us load the dependencies, configure the OpenRouter API to access GPT-5, and prepare the utility functions for image processing.

In [3]:
# Verification des dependances externes
import importlib

_DEPS_STATUS = {}
try:
    importlib.import_module('PIL')
    _DEPS_STATUS['PIL'] = True
except ImportError:
    _DEPS_STATUS['PIL'] = False
    print(f'WARNING: Pillow non installe - pip install Pillow')

try:
    importlib.import_module('requests')
    _DEPS_STATUS['requests'] = True
except ImportError:
    _DEPS_STATUS['requests'] = False
    print(f'WARNING: requests non installe - pip install requests')

try:
    importlib.import_module('matplotlib')
    _DEPS_STATUS['matplotlib'] = True
except ImportError:
    _DEPS_STATUS['matplotlib'] = False
    print(f'WARNING: matplotlib non installe - pip install matplotlib')

try:
    importlib.import_module('dotenv')
    _DEPS_STATUS['dotenv'] = True
except ImportError:
    _DEPS_STATUS['dotenv'] = False
    print(f'WARNING: python-dotenv non installe - pip install python-dotenv')

_all_deps_ok = all(_DEPS_STATUS.values())
if not _all_deps_ok:
    missing = [k for k, v in _DEPS_STATUS.items() if not v]
    print(f'Dependances manquantes: {missing}')
else:
    print('Toutes les dependances sont disponibles')

# Setup environnement et imports
import os
import sys
import json
import requests
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Any, Optional, Union
import base64
from io import BytesIO
from PIL import Image
import matplotlib.pyplot as plt
import logging
from urllib.parse import urlparse

# Chargement robuste de la configuration .env
from dotenv import load_dotenv

current_path = Path.cwd()
env_loaded = False
for _ in range(10):
    env_path = current_path / ".env"
    if env_path.exists():
        load_dotenv(env_path)
        print(f".env charge depuis: {env_path.name}")
        env_loaded = True
        break
    if current_path.name == "GenAI" or len(current_path.parts) <= 1:
        break
    current_path = current_path.parent
if not env_loaded:
    print("WARNING: .env non trouve, utilisation variables environnement")

# GENAI_ROOT pointe vers le dossier GenAI
HELPERS_PATH = current_path / 'shared' / 'helpers'
if HELPERS_PATH.exists():
    sys.path.insert(0, str(HELPERS_PATH.parent))
    try:
        from helpers.genai_helpers import setup_genai_logging, load_genai_config
        print("Helpers GenAI importes")
    except ImportError:
        print("Helpers GenAI non disponibles - mode autonome")

# Configuration logging
logging.basicConfig(level=getattr(logging, debug_level))
logger = logging.getLogger('gpt5_multimodal')

print(f"GPT-5 Multimodal - Analyse et Generation d'Images")
print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Mode: {notebook_mode}, Analyse: {analysis_mode}, Niveau: {educational_level}")
print(f"Langue: {language}, Max tokens: {max_tokens}")

Toutes les dependances sont disponibles


Helpers GenAI importes
GPT-5 Multimodal - Analyse et Generation d'Images
2026-10-29 20:20:32
Mode: interactive, Analyse: detailed, Niveau: university
Langue: francais, Max tokens: 4000


The environment is initialized. The next cell configures access to the OpenRouter API and checks that the GPT-5 model is available for multimodal requests.


### Reading the result — environment initialization

The previous cell attests that the four critical dependencies are resolved on the host machine: **Pillow** (base64 image encoding), **requests** (POST `/chat/completions` to OpenRouter), **matplotlib** (optional local image visualization) and **python-dotenv** (loading the `.env` file). The message `Toutes les dependances sont disponibles` is the precondition for the multimodal pipeline — without it, the call to GPT-5 would raise an `ImportError` at base64 encoding time, not at HTTP request time.

The next block loads the user configuration from `.env` (key `OPENROUTER_API_KEY`, base URL, additional variables). The final display summarizes the notebook parameters: `Mode: interactive, Analyse: detailed, Niveau: university, Langue: francais, Max tokens: 4000`. These values are the ones the following cells will pass as arguments to `analyze_image_with_gpt5()` — any change to a parameter in cell 1 propagates here without re-running the rest of the notebook.

In [4]:
# Configuration API OpenRouter pour GPT-5
print("\n🔑 CONFIGURATION GPT-5 MULTIMODAL")
print("=" * 42)

# Vérification clé API
openrouter_key = os.getenv('OPENROUTER_API_KEY') or os.getenv('OPENAI_API_KEY')
if not openrouter_key:
    # Fallback pour validation structurelle sans clé
    if notebook_mode == "batch" and not analyze_images:
        print("⚠️  Mode batch sans analyse : Clé API ignorée")
        openrouter_key = "dummy_key_for_validation"
    else:
        raise ValueError(
            "OPENROUTER_API_KEY manquante dans .env\n"
            "Configurez votre clé dans le fichier .env (voir .env.example)"
        )

print(f"✅ Clé API OpenRouter configurée")

# Configuration headers et endpoint
api_base_url = os.getenv("OPENROUTER_BASE_URL", os.getenv("OPENAI_BASE_URL", "https://openrouter.ai/api/v1"))
headers = {
    "Authorization": f"Bearer {openrouter_key}",
    "HTTP-Referer": "https://coursia.myia.io",
    "X-Title": "CoursIA GenAI Images - GPT-5 Multimodal",
    "Content-Type": "application/json"
}

# Test connexion et vérification modèle GPT-5
if openrouter_key != "dummy_key_for_validation":
    try:
        response = requests.get(f"{api_base_url}/models", headers=headers, timeout=10)
        if response.status_code == 200:
            models_data = response.json()
            gpt5_models = [m for m in models_data.get('data', []) if 'gpt-5' in m.get('id', '').lower()]
            
            if gpt5_models:
                print(f"✅ Connexion réussie - {len(gpt5_models)} modèles GPT-5 disponibles")
                
                for model in gpt5_models:
                    print(f"  🧠 {model['id']} - Contexte: {model.get('context_length', 'N/A')} tokens")
                    if 'vision' in model.get('capabilities', []):
                        print(f"     👁️  Capacités vision activées")
            else:
                print(f"⚠️  Aucun modèle GPT-5 détecté - vérifiez votre accès")
                print(f"🔍 Modèles disponibles avec 'gpt' : {len([m for m in models_data.get('data', []) if 'gpt' in m.get('id', '').lower()])}")
        else:
            print(f"⚠️  Connexion API: HTTP {response.status_code}")
    except Exception as e:
        print(f"❌ Erreur connexion: {str(e)[:100]}...")
else:
    print("⏭️  Test connexion API sauté (dummy key)")
    
print(f"\n🎯 Modèle sélectionné: {model_name}")
print(f"⚙️  Paramètres: Temperature={temperature}, Max tokens={max_tokens}")
print(f"📊 Mode d'analyse: {analysis_mode}")



🔑 CONFIGURATION GPT-5 MULTIMODAL
✅ Clé API OpenRouter configurée


✅ Connexion réussie - 46 modèles GPT-5 disponibles
  🧠 openai/gpt-5.6-luna-pro - Contexte: 1050000 tokens
  🧠 openai/gpt-5.6-luna-pro:batch - Contexte: 1050000 tokens
  🧠 openai/gpt-5.6-luna - Contexte: 1050000 tokens
  🧠 openai/gpt-5.6-luna:batch - Contexte: 1050000 tokens
  🧠 openai/gpt-5.6-terra-pro - Contexte: 1050000 tokens
  🧠 openai/gpt-5.6-terra-pro:batch - Contexte: 1050000 tokens
  🧠 openai/gpt-5.6-terra - Contexte: 1050000 tokens
  🧠 openai/gpt-5.6-terra:batch - Contexte: 1050000 tokens
  🧠 openai/gpt-5.6-sol-pro - Contexte: 1050000 tokens
  🧠 openai/gpt-5.6-sol-pro:batch - Contexte: 1050000 tokens
  🧠 openai/gpt-5.6-sol - Contexte: 1050000 tokens
  🧠 openai/gpt-5.6-sol:batch - Contexte: 1050000 tokens
  🧠 openai/gpt-5.5-pro - Contexte: 1050000 tokens
  🧠 openai/gpt-5.5-pro:batch - Contexte: 1050000 tokens
  🧠 openai/gpt-5.5 - Contexte: 1050000 tokens
  🧠 openai/gpt-5.5:batch - Contexte: 1050000 tokens
  🧠 openai/gpt-5.4-image-2 - Contexte: 272000 tokens
  🧠 openai/gpt-5.4-n

The API is configured and the connection validated. The next cell defines the multimodal analysis functions that wrap the GPT-5 calls with base64 image encoding.


### Reading the result — OpenRouter connection

The previous cell performed a `GET /models` on the configured OpenRouter endpoint. The `HTTP/1.1 200 OK` response confirms that the API key is valid and that the gateway recognizes the account. The `gpt5_models` extraction filters the models whose identifier contains `gpt-5` — the count depends on the gateway catalogue at call time (the output above displays the complete inventory, on the order of fifty at the reference execution). This is the most visible gap between the official nomenclature (OpenAI publishes `gpt-5`, `gpt-5-mini`, `gpt-5-pro`, etc.) and the gateway nomenclature (each provider appears with its prefix: `openai/gpt-5`, `openai/gpt-5.5-pro`, `openai/gpt-5.4-image-2`, `openai/gpt-5.3-codex`.... This taxonomic explosion is the effect of OpenRouter's strategy of exposing one identifier per `(provider, variant)` pair.

Each `Contexte: N tokens` line of the output gives the model's window, and it is that line you should read model by model: the window follows no per-suffix rule. The output above shows the range — from `gpt-5.2-chat` (128,000 tokens) to the `gpt-5.6`/`gpt-5.5`/`gpt-5.4` generations and their `-pro` variants (million-token window) — and debunks family intuitions: `gpt-5.2-pro` and `gpt-5-pro` stay at 400,000 while `gpt-5.5-pro` reaches 1,050,000; on the image side, `gpt-5-image` shows 400,000 while only the `gpt-5.4-image-2` variant drops to 272,000. An image message plus a long multimodal analysis prompt fit within each of these windows, and the million-token windows open the way to multi-turn workflows where the preliminary analysis is recalled in each subsequent question.

In [5]:
# Fonctions utilitaires pour traitement d'images
def encode_image_base64(image_path: Union[str, Path]) -> str:
    """
    Encode une image locale en base64 pour GPT-5.
    
    Args:
        image_path: Chemin vers l'image locale
        
    Returns:
        String base64 de l'image
    """
    try:
        with open(image_path, "rb") as image_file:
            return base64.b64encode(image_file.read()).decode('utf-8')
    except Exception as e:
        raise ValueError(f"Erreur lecture image {image_path}: {str(e)}")

def download_and_encode_image(image_url: str) -> str:
    """
    Télécharge et encode une image depuis URL.
    
    Args:
        image_url: URL de l'image
        
    Returns:
        String base64 de l'image
    """
    try:
        response = requests.get(image_url, timeout=30)
        response.raise_for_status()
        return base64.b64encode(response.content).decode('utf-8')
    except Exception as e:
        raise ValueError(f"Erreur téléchargement {image_url}: {str(e)}")

def prepare_image_for_gpt5(image_source: Union[str, Path]) -> Dict[str, str]:
    """
    Prépare une image pour GPT-5 (locale ou URL).
    
    Args:
        image_source: Chemin local ou URL de l'image
        
    Returns:
        Dict avec format attendu par GPT-5
    """
    if isinstance(image_source, (str, Path)):
        str_source = str(image_source)
        
        # Vérification URL
        if str_source.startswith(('http://', 'https://')):
            try:
                # Pour les URLs, GPT-5 peut les traiter directement
                return {
                    "type": "image_url",
                    "image_url": {
                        "url": str_source
                    }
                }
            except:
                # Fallback : télécharger et encoder
                base64_image = download_and_encode_image(str_source)
                return {
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:image/jpeg;base64,{base64_image}"
                    }
                }
        else:
            # Image locale
            if Path(str_source).exists():
                base64_image = encode_image_base64(str_source)
                return {
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:image/jpeg;base64,{base64_image}"
                    }
                }
            else:
                raise FileNotFoundError(f"Image non trouvée: {str_source}")
    
    raise ValueError(f"Format d'image non supporté: {type(image_source)}")

print("✅ Fonctions utilitaires images prêtes")

✅ Fonctions utilitaires images prêtes


### Analysis function and demonstration images

The next cell defines the main function `analyze_image_with_gpt5()` which encodes an image in base64 and sends it to the multimodal API. We then prepare a set of test images.


### Reading the result — image encoding pipeline

The previous cell exposes three utility functions that form the encoding pipeline: `encode_image_base64()` (local read + base64 encoding), `download_and_encode_image()` (HTTP GET + encoding for remote URLs) and `prepare_image_for_gpt5()` (dispatcher that chooses between direct URL and base64 based on the prefix). The uniform return is a dict `{type: image_url, image_url: {url: ...}}` which matches the OpenAI specification of the multimodal `content` field — this is the format that `analyze_image_with_gpt5()` will then pass in the `messages` array.

The `print('✅ Fonctions utilitaires images prêtes')` is a loading signal, not an execution test. To verify that the pipeline actually works, one must move on to the analysis cell on a concrete image — the `prepare_image_for_gpt5` function is only validated at that point, and a failure (inaccessible URL, missing file, unsupported format) raises a typed `ValueError` or `FileNotFoundError`.

In [6]:
# Fonction principale d'analyse d'image avec GPT-5
def analyze_image_with_gpt5(image_source: Union[str, Path], 
                           prompt: str = None,
                           analysis_type: str = "detailed") -> Dict[str, Any]:
    """
    Analyse une image avec GPT-5 multimodal.
    
    Args:
        image_source: Chemin local ou URL de l'image
        prompt: Prompt personnalisé (optionnel)
        analysis_type: Type d'analyse ("quick", "detailed", "educational")
        
    Returns:
        Dict avec analyse complète
    """
    
    # Prompts prédéfinis selon le type d'analyse
    analysis_prompts = {
        "quick": f"Décris brièvement cette image en {language}. Sois concis mais précis.",
        
        "detailed": f"""Analyse cette image en détail en {language}. Inclus :
        
1. **Description générale** : Que voit-on dans l'image ?
2. **Éléments visuels** : Couleurs, composition, style artistique
3. **Contexte** : Époque, lieu, situation probable
4. **Détails techniques** : Qualité, résolution apparente, type de photo/illustration
5. **Émotions/Atmosphère** : Quelle ambiance dégage l'image ?
6. **Interprétation** : Signification possible, message artistique

Sois précis et pédagogique dans tes explications.""",
        
        "educational": f"""Tu es un professeur expert analysant cette image pour des étudiants de niveau {educational_level}. En {language}, fournis :

🎯 **ANALYSE PÉDAGOGIQUE**

**1. Description accessible**
- Que montre cette image de façon simple et claire ?

**2. Éléments à observer**
- Quels détails importants les étudiants doivent-ils remarquer ?
- Techniques artistiques ou photographiques utilisées

**3. Contexte éducatif**
- Dans quel domaine d'étude cette image serait-elle utile ?
- Quelles disciplines académiques peuvent l'utiliser ?

**4. Questions pour réflexion**
- 3 questions que tu poserais aux étudiants sur cette image

**5. Connexions interdisciplinaires**
- Comment cette image se connecte-t-elle à d'autres sujets ?

Adapte ton vocabulaire au niveau {educational_level}."""
    }
    
    # Sélection du prompt
    if prompt is None:
        prompt = analysis_prompts.get(analysis_type, analysis_prompts["detailed"])
    
    try:
        print(f"\n🔍 Analyse en cours...")
        print(f"📝 Type: {analysis_type}")
        print(f"🖼️  Source: {str(image_source)[:100]}{'...' if len(str(image_source)) > 100 else ''}")
        
        # Préparation de l'image
        image_data = prepare_image_for_gpt5(image_source)
        
        # Construction du message multimodal
        messages = [
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": prompt
                    },
                    image_data
                ]
            }
        ]
        
        # Payload de la requête
        payload = {
            "model": model_name,
            "messages": messages,
            "max_tokens": max_tokens,
            "temperature": temperature,
            "top_p": top_p
        }
        
        # Requête API
        start_time = datetime.now()
        response = requests.post(
            f"{api_base_url}/chat/completions",
            headers=headers,
            json=payload,
            timeout=120
        )
        response_time = (datetime.now() - start_time).total_seconds()
        
        if response.status_code == 200:
            result = response.json()
            
            analysis_text = result["choices"][0]["message"]["content"]
            
            # Métadonnées de l'analyse
            metadata = {
                "model": model_name,
                "analysis_type": analysis_type,
                "educational_level": educational_level,
                "language": language,
                "timestamp": datetime.now().isoformat(),
                "response_time": response_time,
                "tokens_used": result.get("usage", {}),
                "image_source": str(image_source),
                "prompt_length": len(prompt)
            }
            
            return {
                "success": True,
                "analysis": analysis_text,
                "metadata": metadata,
                "image_source": image_source,
                "analysis_type": analysis_type
            }
        else:
            error_data = response.json() if response.headers.get('content-type', '').startswith('application/json') else {}
            error_msg = error_data.get("error", {}).get("message", f"HTTP {response.status_code}")
            
            return {
                "success": False,
                "error": error_msg,
                "image_source": image_source,
                "status_code": response.status_code
            }
            
    except Exception as e:
        return {
            "success": False,
            "error": str(e),
            "image_source": image_source
        }

print("✅ Fonction d'analyse GPT-5 prête")

✅ Fonction d'analyse GPT-5 prête


The example images and the pedagogical prompts are defined. The next cell launches the demonstration analysis by testing the three modes (quick, detailed, educational) on a concrete image.


### Reading the result — function `analyze_image_with_gpt5`

The previous cell assembles three predefined prompts (`quick`, `detailed`, `educational`) selected by the `analysis_type` argument. The `educational` prompt is the most structuring: it imposes the role, the educational level (read from `educational_level`), the language (`language`), and an output format with five explicit sections (`Description accessible`, `Éléments à observer`, `Contexte éducatif`, `Questions pour réflexion`, `Connexions interdisciplinaires`). This rigidity of the prompt is what guarantees the reproducibility of the analyses between two sessions.

The HTTP call block measures the response time (`response_time`) and retrieves `usage` which contains `prompt_tokens`, `completion_tokens` and `total_tokens`. The `reasoning_tokens` field (present in the GPT-5 image-generation output) reflects the tokens consumed in the internal reasoning phase — an invisible cost in `completion_tokens` alone, but billed. The next cell uses this instrumentation to compare the three modes on the same image.

In [7]:
# Exemples d'images pour démonstration
print("\n🖼️  EXEMPLES D'ANALYSE GPT-5")
print("=" * 40)

# Images d'exemple (URLs publiques pour tests)
example_images = [
    {
        "title": "🏛️ Architecture Historique",
        "url": "https://images.unsplash.com/photo-1539037116277-4db20889f2d4?w=800",  # Colisée Rome
        "description": "Monument historique romain - idéal pour analyse architecturale",
        "category": "Histoire/Architecture"
    },
    {
        "title": "🔬 Science et Technologie",
        "url": "https://images.unsplash.com/photo-1532094349884-543bc11b234d?w=800",  # Laboratoire
        "description": "Environnement scientifique - parfait pour analyse technique",
        "category": "Science/Technologie"
    },
    {
        "title": "🎨 Art et Culture",
        "url": "https://images.unsplash.com/photo-1541961017774-22349e4a1262?w=800",  # Peinture
        "description": "Œuvre artistique - excellent pour analyse esthétique",
        "category": "Art/Culture"
    },
    {
        "title": "🌍 Nature et Environnement",
        "url": "https://images.unsplash.com/photo-1506905925346-21bda4d32df4?w=800",  # Paysage naturel
        "description": "Paysage naturel - parfait pour analyse géographique",
        "category": "Géographie/Environnement"
    }
]

# Affichage des exemples
for i, example in enumerate(example_images, 1):
    print(f"\n{i}. {example['title']}")
    print(f"   📂 Catégorie: {example['category']}")
    print(f"   📝 {example['description']}")
    print(f"   🔗 URL: {example['url'][:60]}...")

print(f"\n💡 Conseils pour l'analyse avec GPT-5:")
print(f"• Utilisez des images de haute qualité")
print(f"• Posez des questions spécifiques")
print(f"• Exploitez le contexte éducatif")
print(f"• Combinez analyse textuelle et visuelle")
print(f"• Adaptez le niveau de complexité")


🖼️  EXEMPLES D'ANALYSE GPT-5

1. 🏛️ Architecture Historique
   📂 Catégorie: Histoire/Architecture
   📝 Monument historique romain - idéal pour analyse architecturale
   🔗 URL: https://images.unsplash.com/photo-1539037116277-4db20889f2d4...

2. 🔬 Science et Technologie
   📂 Catégorie: Science/Technologie
   📝 Environnement scientifique - parfait pour analyse technique
   🔗 URL: https://images.unsplash.com/photo-1532094349884-543bc11b234d...

3. 🎨 Art et Culture
   📂 Catégorie: Art/Culture
   📝 Œuvre artistique - excellent pour analyse esthétique
   🔗 URL: https://images.unsplash.com/photo-1541961017774-22349e4a1262...

4. 🌍 Nature et Environnement
   📂 Catégorie: Géographie/Environnement
   📝 Paysage naturel - parfait pour analyse géographique
   🔗 URL: https://images.unsplash.com/photo-1506905925346-21bda4d32df4...

💡 Conseils pour l'analyse avec GPT-5:
• Utilisez des images de haute qualité
• Posez des questions spécifiques
• Exploitez le contexte éducatif
• Combinez analyse textuell

### Test : analysis of a demonstration image

Let us test the analysis function on a concrete image. GPT-5 will describe the content, identify the visual elements, and provide a structured analysis.

In [8]:
# Analyse d'une image de démonstration
print("\n🚀 ANALYSE DE DÉMONSTRATION - GPT-5")
print("=" * 50)

# Sélection d'une image pour la démonstration
selected_example = example_images[0]  # Architecture historique
print(f"🎯 Analyse: {selected_example['title']}")
print(f"📂 Catégorie: {selected_example['category']}")

# Test de l'analyse avec les différents modes
analysis_results = []

for mode in ['quick', 'detailed', 'educational']:
    print(f"\n🔍 Mode d'analyse: {mode.upper()}")
    print("-" * 30)
    
    # Analyse de l'image
    result = analyze_image_with_gpt5(
        image_source=selected_example['url'],
        analysis_type=mode
    )
    
    if result['success']:
        print(f"✅ Analyse {mode} réussie")
        print(f"⏱️  Temps: {result['metadata']['response_time']:.2f}s")
        print(f"🔢 Tokens: {result['metadata']['tokens_used']}")
        
        # Affichage de l'analyse (tronquée pour la démo)
        analysis_preview = result['analysis'][:300] + "..." if len(result['analysis']) > 300 else result['analysis']
        print(f"\n📝 **Aperçu de l'analyse {mode}:**")
        print(analysis_preview)
        
        analysis_results.append(result)
    else:
        print(f"❌ Échec analyse {mode}: {result['error']}")
    
    print()  # Séparation

# Comparaison des résultats
if analysis_results:
    print(f"\n📊 COMPARAISON DES MODES D'ANALYSE")
    print("=" * 45)
    
    for result in analysis_results:
        mode = result['analysis_type']
        length = len(result['analysis'])
        tokens = result['metadata'].get('tokens_used', {}).get('total_tokens', 'N/A')
        time = result['metadata']['response_time']
        
        print(f"{mode.capitalize():12} | {length:4d} chars | {tokens:>6} tokens | {time:5.2f}s")
    
    print(f"\n💡 Observations:")
    print(f"• Mode 'quick': Réponses concises et rapides")
    print(f"• Mode 'detailed': Analyse approfondie et structurée")
    print(f"• Mode 'educational': Adapté à l'enseignement avec questions")
else:
    print(f"\n⚠️  Aucune analyse réussie - vérifiez votre configuration")


🚀 ANALYSE DE DÉMONSTRATION - GPT-5
🎯 Analyse: 🏛️ Architecture Historique
📂 Catégorie: Histoire/Architecture

🔍 Mode d'analyse: QUICK
------------------------------

🔍 Analyse en cours...
📝 Type: quick
🖼️  Source: https://images.unsplash.com/photo-1539037116277-4db20889f2d4?w=800


✅ Analyse quick réussie
⏱️  Temps: 6.50s
🔢 Tokens: {'prompt_tokens': 653, 'completion_tokens': 226, 'total_tokens': 879, 'cost': 0, 'is_byok': True, 'prompt_tokens_details': {'cached_tokens': 0, 'cache_write_tokens': 0, 'audio_tokens': 0, 'video_tokens': 0}, 'cost_details': {'upstream_inference_cost': 0.00307625, 'upstream_inference_prompt_cost': 0.00081625, 'upstream_inference_completions_cost': 0.00226}, 'completion_tokens_details': {'reasoning_tokens': 128, 'image_tokens': 0, 'audio_tokens': 0}}

📝 **Aperçu de l'analyse quick:**
Vue au coucher du soleil sur la Gran Vía de Madrid : immeubles historiques illuminés, circulation légère, ciel orangé en arrière-plan, avec au premier plan l’édifice Metrópolis à dôme noir et doré.


🔍 Mode d'analyse: DETAILED
------------------------------

🔍 Analyse en cours...
📝 Type: detailed
🖼️  Source: https://images.unsplash.com/photo-1539037116277-4db20889f2d4?w=800


✅ Analyse detailed réussie
⏱️  Temps: 50.00s
🔢 Tokens: {'prompt_tokens': 758, 'completion_tokens': 1786, 'total_tokens': 2544, 'cost': 0, 'is_byok': True, 'prompt_tokens_details': {'cached_tokens': 0, 'cache_write_tokens': 0, 'audio_tokens': 0, 'video_tokens': 0}, 'cost_details': {'upstream_inference_cost': 0.0188075, 'upstream_inference_prompt_cost': 0.0009475, 'upstream_inference_completions_cost': 0.01786}, 'completion_tokens_details': {'reasoning_tokens': 960, 'image_tokens': 0, 'audio_tokens': 0}}

📝 **Aperçu de l'analyse detailed:**
1) Description générale
- Vue urbaine prise en hauteur au coucher du soleil. 
- Une large avenue traverse l’image en diagonale, bordée d’immeubles élégants aux façades claires.
- Au premier plan à gauche, un bâtiment d’angle coiffé d’un dôme sombre orné d’une statue. À droite, un autre immeuble cour...


🔍 Mode d'analyse: EDUCATIONAL
------------------------------

🔍 Analyse en cours...
📝 Type: educational
🖼️  Source: https://images.unsplash.com/photo

✅ Analyse educational réussie
⏱️  Temps: 21.12s
🔢 Tokens: {'prompt_tokens': 804, 'completion_tokens': 1522, 'total_tokens': 2326, 'cost': 0, 'is_byok': True, 'prompt_tokens_details': {'cached_tokens': 0, 'cache_write_tokens': 0, 'audio_tokens': 0, 'video_tokens': 0}, 'cost_details': {'upstream_inference_cost': 0.016225, 'upstream_inference_prompt_cost': 0.001005, 'upstream_inference_completions_cost': 0.01522}, 'completion_tokens_details': {'reasoning_tokens': 704, 'image_tokens': 0, 'audio_tokens': 0}}

📝 **Aperçu de l'analyse educational:**
🎯 ANALYSE PÉDAGOGIQUE

1) Description accessible
- Une vue urbaine au coucher du soleil, prise en hauteur, montrant une grande avenue bordée d’immeubles historiques. Les façades claires, richement décorées, sont éclairées, la circulation est modérée et l’horizon révèle une chaîne de montagnes sous u...


📊 COMPARAISON DES MODES D'ANALYSE
Quick        |  197 chars |    879 tokens |  6.50s
Detailed     | 3183 chars |   2544 tokens | 50.00s
Education

### Reading the result — quick/detailed/educational comparison

The previous cell launches three successive calls on the `Architecture Historique` image (actually the Gran Vía in Madrid — a first factual surprise worth noting). The output ends with a `📊 COMPARAISON DES MODES D'ANALYSE` table printed by the code itself: characters, tokens and time are recapitulated there for the three modes — read it directly in the output, it reflects the execution you have just launched (its values change on each re-run).

**What to read there, as a guide**:
1. The response time is not linear in output length: compare the ratio of the `total_tokens` and that of the times — latency is dominated by the reasoning phase (`reasoning_tokens`, in the `completion_tokens_details` of each block), not by generation itself.
2. The upstream cost (`upstream_inference_cost` in the token details) roughly follows the total tokens; the `educational` prompt is the most expensive because it injects structure + output format into the input.
3. For an `alt-text` use, it is the specialized prompt of the next section that guarantees the constraint (≤ 125 characters imposed in the prompt) — the `quick` output is concise, but it is the prompt that holds the WCAG promise, not the mere brevity of the mode. For a structured pedagogical use, the `educational` mode produces student-guiding questions in addition.

### Guided example: a personalized analysis prompt for the Art domain

*Contribution from a group of EPF students (TP2, PR #20551), integrated as a guided example.*

The three analysis modes (quick, detailed, educational) cover general uses, but real cases often require specialized prompts. This example builds a prompt optimized for the **Art** domain: an expert role, four mandatory sections (technical description, stylistic analysis, context and meaning, guiding questions), a requirement for specific vocabulary, and a factuality instruction that separates what the model sees from what it infers.

**Key points**:
- The prompt imposes the **output format** (numbered sections) and the **language** (`language`) — the answer becomes mechanically checkable
- Quality is not only read, it is **measured by the code**: presence of the sections, detected art vocabulary, number of guiding questions
- An automated evaluation turns an impression ("the answer is good") into a reproducible criterion

In [9]:
# Exemple guide : un prompt d'analyse personnalise pour le domaine Art
# (contribution d'un groupe d'etudiants EPF, TP2, PR #20551)

# Etape 1 : Choix du domaine
mon_domaine = "Art"  # parmi "Histoire", "Sciences", "Geographie", "Art", "Medecine"

# Etape 2 : Creation du prompt personnalise
# Structure suivie : role expert, format de sortie en sections, vocabulaire
# specifique au domaine, questions guides, consigne de factualite
mon_prompt_analyse = f"""Tu es un professeur expert en {mon_domaine}
Réponds en {language}, avec un vocabulaire précis propre au domaine de l'art.

Structure ta réponse avec exactement ces sections :

**1. Description technique**
- Sujet représenté, type d'image (photographie, peinture, architecture...)
- Composition : cadrage, plans, lignes de force, points focaux
- Palette : couleurs dominantes, contrastes, lumière (source, direction, ambiance)

**2. Analyse stylistique**
- Style ou courant artistique reconnaissable (ex. néoclassique, éclectique, Art déco)
- Éléments formels notables : ornementation, symétrie, rythme, perspective
- Utilise au moins 5 termes de vocabulaire spécifique (ex. façade, corniche, dôme, clair-obscur, point de fuite)

**3. Contexte et signification**
- Période et contexte culturel probables
- Message ou émotion que l'image cherche à transmettre

**4. Trois questions guides pour les étudiants**
- Trois questions numérotées, chacune se terminant par un point d'interrogation
- Progression : une question d'observation, une d'analyse, une d'interprétation

Reste factuel : distingue ce que tu vois de ce que tu déduis. """

# Etape 3 : Test sur l'image de demonstration
image_test = example_images[2]['url']  # Peinture abstraite (Art et Culture)
resultat_exo1 = analyze_image_with_gpt5(image_test, prompt=mon_prompt_analyse)

if resultat_exo1['success']:
    texte = resultat_exo1['analysis']
    print(texte)
    print(f"\nTemps : {resultat_exo1['metadata']['response_time']:.1f}s")
else:
    print(f"Échec : {resultat_exo1['error']}")
    texte = ""

# Etape 4 : Evaluation automatique de la qualite
if texte:
    t = texte.lower()

    # La reponse contient-elle bien les sections demandees ?
    sections = ["description technique", "analyse stylistique",
                "contexte", "questions guides"]
    print("\n--- Sections présentes ---")
    for s in sections:
        print(f"{'OK ' if s in t else 'MANQUE'}  {s}")

    # Le vocabulaire est-il adapte au domaine ?
    vocab_art = ["composition", "palette", "lumière", "perspective", "style",
                 "façade", "ornement", "contraste", "symétrie", "plan",
                 "point de fuite", "ligne", "cadrage", "dôme", "corniche"]
    trouves = [v for v in vocab_art if v in t]
    print(f"\n--- Vocabulaire artistique : {len(trouves)}/{len(vocab_art)} termes ---")
    print(", ".join(trouves))

    # Les questions guides sont-elles presentes (3 attendues) ?
    questions = [l.strip() for l in texte.splitlines() if l.strip().endswith("?")]
    print(f"\n--- Questions guides détectées : {len(questions)} (attendu : 3) ---")
    for q in questions:
        print(" •", q)



🔍 Analyse en cours...
📝 Type: detailed
🖼️  Source: https://images.unsplash.com/photo-1541961017774-22349e4a1262?w=800


**1. Description technique**
- Sujet représenté, type d'image: Abstraction non figurative; peinture sur toile. Les coulures nettes, la saturation chromatique et le séchage rapide suggèrent probablement de l’acrylique (déduction).
- Composition: Organisation majoritairement verticale, structurée par trois grandes masses colorées: une zone chaude à droite (oranges/rouges), une nappe froide au centre-gauche (turquoise/vert bleuté), et un champ jaune à gauche. Lignes de force ascendantes et descendantes créées par les traînées et dripping; points focaux situés aux zones de contact chaud/froid, notamment au croisement du turquoise avec l’orange. Plan pictural sans profondeur illusionniste; absence de figure identifiable (constat).
- Palette: Dominantes chaudes (jaune cadmium, orange, vermillon) opposées à un pôle froid (turquoise/vert émeraude), avec rehauts de blanc. Contrastes complémentaires orange/bleu marqués; valeurs globalement hautes. Lumière non référentielle: éclats lumineux dus a

### Practical applications : accessibility and pedagogy

GPT-5 excels in two concrete use cases:
- **Alt text** : automatically generate accessibility descriptions for web images
- **Pedagogical analysis** : use multimodal vision to create educational content from images


### Reading the result — alt text generation

The previous cell calls `analyze_image_with_gpt5()` with a specialized accessibility prompt that imposes three constraints: (1) output ≤ 125 characters, (2) factual and objective tone, (3) no additional formatting. The output obtained describes the scene (aerial view of the Gran Vía in Madrid) in a single factual sentence; the `print` displays its length in characters — check that it remains under the 125 threshold imposed by the prompt (the exact text changes on each execution, the constraint is held by the prompt).

**Why this length matters** : the WCAG 2.1 standard (criterion 1.1.1 *Non-text Content*) recommends short alt texts for informative images. Beyond 125 characters, screen readers truncate and the experience degrades. Below 50, the user loses context. The guided example in cell 19 below runs the three length levels to study this comfort zone — this is one of the uses where native multimodality replaces a separate pipeline (vision model → text model → formatting).

In [10]:
# Génération de descriptions d'accessibilité (Alt text)
if generate_alt_text and analysis_results:
    print("\n♿ GÉNÉRATION DE DESCRIPTIONS D'ACCESSIBILITÉ")
    print("=" * 55)
    
    # Prompt spécialisé pour l'accessibilité
    accessibility_prompt = f"""Génère une description d'accessibilité (alt text) pour cette image en {language}.
    
Critères :
- Maximum 125 caractères
- Description factuelle et objective
- Inclut les éléments essentiels pour la compréhension
- Évite les interprétations subjectives
- Adapté aux lecteurs d'écran

Format : Fournis UNIQUEMENT la description, sans formatage supplémentaire."""
    
    # Génération de l'alt text
    alt_result = analyze_image_with_gpt5(
        image_source=selected_example['url'],
        prompt=accessibility_prompt
    )
    
    if alt_result['success']:
        alt_text = alt_result['analysis'].strip()
        
        print(f"✅ Description d'accessibilité générée")
        print(f"📝 Alt text ({len(alt_text)} caractères) :")
        print(f'   "{alt_text}"')
        
        if len(alt_text) > 125:
            print(f"⚠️  Longueur dépassée ({len(alt_text)}/125 chars) - considérez une version plus courte")
        else:
            print(f"✅ Longueur optimale ({len(alt_text)}/125 chars)")
    else:
        print(f"❌ Erreur génération alt text: {alt_result['error']}")
else:
    print(f"\n⏭️  Génération alt text désactivée ou pas d'analyse disponible")


♿ GÉNÉRATION DE DESCRIPTIONS D'ACCESSIBILITÉ

🔍 Analyse en cours...
📝 Type: detailed
🖼️  Source: https://images.unsplash.com/photo-1539037116277-4db20889f2d4?w=800


✅ Description d'accessibilité générée
📝 Alt text (113 caractères) :
   "Vue aérienne de la Gran Vía à Madrid au coucher du soleil, immeubles éclairés et voitures circulant sur l’avenue."
✅ Longueur optimale (113/125 chars)


### Guided example: multi-level accessibility descriptions

*Contribution from a group of EPF students (TP2, PR #20551), integrated as a guided example.*

Web accessibility requires alternative texts (alt text) adapted to the context. The same image may require descriptions of different lengths depending on the use: quick navigation (≤ 125 characters), detailed description for screen readers (200-400 characters), or a pedagogical summary. This example generates the three levels with three distinct prompts — each one explicitly sets the target length and the intended audience — then measures in code the actual length of each text produced.

**Key points**:
- Each level has its **own prompt**: the maximum length and the audience are spelled out in the instruction
- The check relies on `len()` of the actual texts — compliance with the WCAG limit is **observed**, not assumed
- The short text targets ≤ 125 characters: beyond that, screen readers truncate (see the WCAG analysis above)

In [11]:
# Exemple guide : descriptions d'accessibilite multi-niveaux
# (contribution d'un groupe d'etudiants EPF, TP2, PR #20551)

# Etape 1 : Choix de l'image
mon_image = "https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcSm6AMs5m9c7Nmp-IB1HUY4LeA4EjTF8sUTmW3JU-26cs1BF0cJuaFBV8c&s=10"  # image choisie par le groupe

# Etape 2 : Prompt pour alt text court (<= 125 caracteres)
prompt_court = "Fournissez une description factuelle et objective de l'image, en maximum 125 caractères, sans interprétation."

# Etape 3 : Prompt pour description detaillee (200-400 caracteres)
prompt_detaille = "Fournissez une description détaillée de l'image, en 200 à 400 caractères, pour les lecteurs d'écran. Incluez les éléments essentiels, les couleurs, la disposition et les actions visibles."
# Contraintes imposees : elements essentiels, couleurs, disposition, actions visibles

# Etape 4 : Prompt pour resume pedagogique niveau "secondary"
prompt_pedago = "Fournissez un résumé pédagogique de l'image, adapté à des élèves du secondaire. Utilisez un vocabulaire accessible et posez des questions de réflexion."

# Generation des trois niveaux
resultats = {}
for nom, prompt in [("court", prompt_court), ("detaille", prompt_detaille), ("pedago", prompt_pedago)]:
    r = analyze_image_with_gpt5(mon_image, prompt=prompt)
    if r['success']:
        resultats[nom] = r['analysis']
        print(f"\n--- {nom.upper()} ({len(r['analysis'])} chars) ---")
        print(r['analysis'][:300])

# Etape 5 : Verification des contraintes de longueur
print("\n Vérification des résultats")
print("Nombre de caractères dans le texte court:", len(resultats.get("court", "")))
print("Nombre de caractères dans la description détaillée:", len(resultats.get("detaille", "")))

print("\nRésultats disponibles pour les trois niveaux : court, détaillé et pédagogique:")
print("Texte court (<= 125 chars):", resultats.get("court", "Non généré"))
print("Description détaillée (200-400 chars):", resultats.get("detaille", "Non généré"))
print("Résumé pédagogique:", resultats.get("pedago", "Non généré"))



🔍 Analyse en cours...
📝 Type: detailed
🖼️  Source: https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcSm6AMs5m9c7Nmp-IB1HUY4LeA4EjTF8sUTmW3JU-26cs1B...



--- COURT (93 chars) ---
Personnage 3D en vêtements sombres, gants noirs, tenant un pistolet levé, vu de trois-quarts.

🔍 Analyse en cours...
📝 Type: detailed
🖼️  Source: https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcSm6AMs5m9c7Nmp-IB1HUY4LeA4EjTF8sUTmW3JU-26cs1B...



--- DETAILLE (324 chars) ---
Personnage 3D type agent tactique, vu en buste trois‑quarts. Visage sérieux, cheveux bruns mi‑longs; il regarde vers la gauche. Haut moulant bleu nuit, gants noirs, harnais et étuis. Avant‑bras musclé en évidence, pistolet levé près du visage; fusil en bandoulière. Arrière‑plan industriel sombre, éc

🔍 Analyse en cours...
📝 Type: detailed
🖼️  Source: https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcSm6AMs5m9c7Nmp-IB1HUY4LeA4EjTF8sUTmW3JU-26cs1B...



--- PEDAGO (1494 chars) ---
Résumé pédagogique
- L’image montre un personnage masculin créé en images de synthèse (3D), comme dans un jeu vidéo ou un film d’animation.
- Il porte une tenue tactique sombre avec gants et harnais. Il tient un pistolet, ce qui suggère une scène d’action ou de danger.
- Son visage est sérieux et co

 Vérification des résultats
Nombre de caractères dans le texte court: 93
Nombre de caractères dans la description détaillée: 324

Résultats disponibles pour les trois niveaux : court, détaillé et pédagogique:
Texte court (<= 125 chars): Personnage 3D en vêtements sombres, gants noirs, tenant un pistolet levé, vu de trois-quarts.
Description détaillée (200-400 chars): Personnage 3D type agent tactique, vu en buste trois‑quarts. Visage sérieux, cheveux bruns mi‑longs; il regarde vers la gauche. Haut moulant bleu nuit, gants noirs, harnais et étuis. Avant‑bras musclé en évidence, pistolet levé près du visage; fusil en bandoulière. Arrière‑plan industriel sombre, é

The demonstration analyses are complete. The interactive mode below lets you experiment with your own images and personalized analysis prompts.


### Reading the result — batch versus interactive mode

The previous cell tries `input()` to retrieve a URL or a local path. In Papermill / batch execution, standard input is not available: the `except EOFError` intercepts cleanly and switches to the message `⏭️  Mode interactif non disponible (exécution automatisée)`. This pattern is the repository's convention for notebooks that mix interactive and batch use: the cell remains functional in both modes without `raise NotImplementedError` (forbidden by C.1 — `pass`/`print` are the correct stubs).

The next cell moves on to the application use cases — this is the bridge between the technical demonstration and the business use. The 5 domains listed (History, Science, Geography, Art, Medicine) cover the disciplines where multimodal analysis brings the most versus a purely textual model.

In [12]:
# Mode interactif - Analyse d'image personnalisée
if notebook_mode == "interactive" and not skip_widgets:
    print("\n🎨 MODE INTERACTIF - ANALYSE PERSONNALISÉE")
    print("=" * 55)
    
    print("\n💡 Analysez votre propre image avec GPT-5:")
    print("Formats supportés: URL https:// ou chemin local")
    print("(Laissez vide pour passer à la suite)")
    
    try:
        user_image = input("\n🖼️  URL ou chemin de votre image: ").strip()

        if user_image:
            # Paramètres d'analyse personnalisés
            print("\n⚙️  Paramètres d'analyse (appuyez Entrée pour défaut):")
            custom_mode = input(f"📊 Mode [{analysis_mode}]: ").strip() or analysis_mode
            custom_prompt = input("📝 Prompt personnalisé (optionnel): ").strip()

            print(f"\n🔍 Analyse de votre image en cours...")

            # Analyse personnalisée
            if custom_prompt:
                user_result = analyze_image_with_gpt5(
                    image_source=user_image,
                    prompt=custom_prompt
                )
            else:
                user_result = analyze_image_with_gpt5(
                    image_source=user_image,
                    analysis_type=custom_mode
                )

            if user_result['success']:
                print(f"\n🎉 Analyse réussie!")
                print(f"⏱️  Temps: {user_result['metadata']['response_time']:.2f}s")
                print(f"\n📝 **Analyse GPT-5:**")
                print(user_result['analysis'])

                # Option de sauvegarde
                if export_analysis:
                    try:
                        save_choice = input("\n💾 Sauvegarder cette analyse ? (o/N): ").strip().lower()
                        if save_choice in ['o', 'oui', 'y', 'yes']:
                            output_dir = current_path / 'outputs' / 'gpt5_analysis'
                            output_dir.mkdir(parents=True, exist_ok=True)

                            timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
                            analysis_file = output_dir / f"gpt5_analysis_{timestamp}.json"

                            with open(analysis_file, 'w', encoding='utf-8') as f:
                                json.dump(user_result, f, indent=2, ensure_ascii=False)

                            print(f"💾 Analyse sauvegardée: {analysis_file}")
                    except Exception as save_err:
                        if "StdinNotImplemented" in type(save_err).__name__:
                            print("\n⏭️  Sauvegarde automatique ignorée en mode batch")
                        else:
                            raise
            else:
                print(f"\n❌ Erreur: {user_result['error']}")
                print(f"🔍 Source: {user_result['image_source']}")
        else:
            print("\n⏭️  Mode interactif ignoré")

    except (KeyboardInterrupt, EOFError) as e:
        # Gestion interruption normale
        print(f"\n⏭️  Mode interactif interrompu ({type(e).__name__})")
    except Exception as e:
        # Gestion des erreurs d'input en mode non-interactif (Papermill, etc.)
        error_type = type(e).__name__
        if "StdinNotImplemented" in error_type or "input" in str(e).lower():
            print("\n⏭️  Mode interactif non disponible (exécution automatisée)")
        else:
            # Autre erreur inattendue - afficher pour débogage
            print(f"\n⚠️  Erreur inattendue: {error_type} - {str(e)[:100]}")
            print("⏭️  Passage à la suite du notebook")
else:
    print("\n🤖 Mode batch - Interface interactive désactivée")
    print("💡 Pour mode interactif: notebook_mode = 'interactive'")


🎨 MODE INTERACTIF - ANALYSE PERSONNALISÉE

💡 Analysez votre propre image avec GPT-5:
Formats supportés: URL https:// ou chemin local
(Laissez vide pour passer à la suite)

⏭️  Mode interactif non disponible (exécution automatisée)


With the interactive mode handled, the next cell presents the main application use cases of visual analysis by GPT-5 in a professional and pedagogical context.


### Reading the result — domain × prompt matrix

The previous cell catalogs five application domains (History, Science, Geography, Art and Culture, Medicine) with for each a predefined `prompt_template` that can be passed directly to `analyze_image_with_gpt5()`. Each template imposes a disciplinary role, an output format, and a tone suited to the domain — for example, the `Médecine` prompt specifies *Analyse cette image médicale pour des étudiants en médecine. Explique l'anatomie visible et les points d'intérêt clinique.*

The output contains no visible GPT-5 output — the cell only *announced* the use cases, without executing them yet. The guided example that follows runs one of them for geography, and the exercises at the end of the notebook reuse the same analysis function on other questions: stability of a repeated response, fidelity of an alt text, structured scoring in JSON. This is an interesting pedagogical inversion: cell 24 shows the templates, the exercises ask to apply the analysis function to other questions.

In [13]:
# Cas d'usage pédagogiques avec GPT-5
print("\n🎓 CAS D'USAGE PÉDAGOGIQUES GPT-5")
print("=" * 45)

# Exemples d'applications éducatives
educational_use_cases = {
    "Histoire": {
        "description": "Analyse de documents historiques, œuvres d'art, monuments",
        "exemple": "Analyser une fresque renaissance pour comprendre le contexte social",
        "prompt_template": "Analyse cette image historique pour des étudiants en histoire. Explique le contexte historique, les éléments symboliques, et l'importance culturelle."
    },
    
    "Sciences": {
        "description": "Analyse d'expériences, schémas scientifiques, phénomènes naturels",
        "exemple": "Expliquer un diagramme de cellule ou une réaction chimique",
        "prompt_template": "En tant qu'enseignant de sciences, explique cette image scientifique. Identifie les éléments techniques et leur fonctionnement."
    },
    
    "Géographie": {
        "description": "Étude de paysages, cartes, phénomènes géologiques",
        "exemple": "Analyser une photo satellite ou un paysage géographique",
        "prompt_template": "Analyse cette image géographique pour des étudiants. Explique les formations géologiques, le climat, et l'impact humain visible."
    },
    
    "Art et Culture": {
        "description": "Critique artistique, analyse stylistique, histoire de l'art",
        "exemple": "Comprendre les techniques d'un tableau impressionniste",
        "prompt_template": "Fais une analyse artistique de cette œuvre pour des étudiants en art. Inclus style, techniques, et signification culturelle."
    },
    
    "Médecine": {
        "description": "Analyse d'imagerie médicale, anatomie, cas cliniques",
        "exemple": "Expliquer une radiographie ou un schéma anatomique",
        "prompt_template": "Analyse cette image médicale pour des étudiants en médecine. Explique l'anatomie visible et les points d'intérêt clinique."
    }
}

# Affichage des cas d'usage
for domain, info in educational_use_cases.items():
    print(f"\n📚 **{domain}**")
    print(f"   📋 {info['description']}")
    print(f"   💡 Exemple: {info['exemple']}")
    print(f"   📝 Template de prompt disponible")

print(f"\n🎯 Avantages GPT-5 pour l'éducation:")
print(f"• **Multimodal** : Analyse texte + image simultanée")
print(f"• **Contextuel** : Comprend le niveau éducatif")
print(f"• **Adaptable** : Ajuste le vocabulaire selon l'audience")
print(f"• **Interactif** : Permet les questions de suivi")
print(f"• **Précis** : Analyse détaillée et factuelle")

print(f"\n📝 Exemple de workflow pédagogique:")
print(f"1. Sélection d'image pertinente au cours")
print(f"2. Analyse GPT-5 avec prompt éducatif")
print(f"3. Génération de questions pour les étudiants")
print(f"4. Discussion interactive basée sur l'analyse")
print(f"5. Évaluation de la compréhension")


🎓 CAS D'USAGE PÉDAGOGIQUES GPT-5

📚 **Histoire**
   📋 Analyse de documents historiques, œuvres d'art, monuments
   💡 Exemple: Analyser une fresque renaissance pour comprendre le contexte social
   📝 Template de prompt disponible

📚 **Sciences**
   📋 Analyse d'expériences, schémas scientifiques, phénomènes naturels
   💡 Exemple: Expliquer un diagramme de cellule ou une réaction chimique
   📝 Template de prompt disponible

📚 **Géographie**
   📋 Étude de paysages, cartes, phénomènes géologiques
   💡 Exemple: Analyser une photo satellite ou un paysage géographique
   📝 Template de prompt disponible

📚 **Art et Culture**
   📋 Critique artistique, analyse stylistique, histoire de l'art
   💡 Exemple: Comprendre les techniques d'un tableau impressionniste
   📝 Template de prompt disponible

📚 **Médecine**
   📋 Analyse d'imagerie médicale, anatomie, cas cliniques
   💡 Exemple: Expliquer une radiographie ou un schéma anatomique
   📝 Template de prompt disponible

🎯 Avantages GPT-5 pour l'éducat

### Guided example: three-mode personalized analysis for a domain

*Contribution from a group of EPF students (TP2, PR #20551), integrated as a guided example.*

Synthesis of the two previous examples: the `analyse_complete()` function chains three analyses of the same image — a short factual description, a detailed description for screen readers, a pedagogical summary contextualized by the domain — and returns everything in a dictionary displayed with `pprint`. The domain (`Géographie`), the level (`university`) and the image are set at the top of the cell.

**Key points**:
- A single function call produces the **three levels** of description — the result is a reusable dictionary, not three separate displays
- The educational prompt is **built from the domain**: changing `domain` at the top of the cell adapts the analysis to another discipline

In [14]:
# Exemple guide : analyse personnalisee a trois modes pour un domaine
# (contribution d'un groupe d'etudiants EPF, TP2, PR #20551)

# 1. Sélection du domaine et configuration
domain = "Géographie"
niveau = "university"  # Niveau éducatif cible

# 2. Image à analyser (URL publique)
image_source = "https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcRPQ_gGhRNQdk5gxBqyA9diGOVps2ZliOpWX2ifjfPSPg&s=10"  # carte de France (choix du groupe)

# 3. Création du prompt personnalisé (inspiré des prompt_template de educational_use_cases)
custom_prompt = f"""Tu es un professeur expert en {domain}.
Analyse cette image pour des étudiants de niveau {niveau}.
Inclus :
1. Description accessible du sujet
2. Éléments techniques importants à observer
3. Contexte historique ou scientifique
4. 3 questions de réflexion pour les étudiants
"""

# 4. Analyse avec les 3 modes de description
def analyse_complete(image_url, prompt, domain):
    """
    Analyse une image avec les 3 modes et retourne les résultats.

    Args:
        image_url: URL ou chemin de l'image
        prompt: Prompt personnalisé pour le mode educational
        domain: Domaine d'étude pour le contexte

    Returns:
        Dict avec les 3 analyses (quick, detailed, educational)
    """



    quick = "Fournissez une description factuelle et objective de l'image, en maximum 125 caractères, sans interprétation."
    detailed = "Fournissez une description détaillée de l'image, en 200 à 400 caractères, pour les lecteurs d'écran. Incluez les éléments essentiels, les couleurs, la disposition et les actions visibles."
    educational = f"""Fournissez un résumé pédagogique de l'image, adapté à des élèves du {domain}. Utilisez un vocabulaire accessible et posez des questions de réflexion."""

    results = {}
    for nom, prompt in [("quick", quick), ("detailed", detailed), ("educational", educational)]:
        r = analyze_image_with_gpt5(image_source, prompt=prompt)
        if r['success']:
          results[nom] = r['analysis']
    return results

# 5. Exécuter l'analyse et afficher les résultats
import pprint
results = analyse_complete(image_source, custom_prompt, domain)
pprint.pprint(results)



🔍 Analyse en cours...
📝 Type: detailed
🖼️  Source: https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcRPQ_gGhRNQdk5gxBqyA9diGOVps2ZliOpWX2ifjfPSPg&s...



🔍 Analyse en cours...
📝 Type: detailed
🖼️  Source: https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcRPQ_gGhRNQdk5gxBqyA9diGOVps2ZliOpWX2ifjfPSPg&s...



🔍 Analyse en cours...
📝 Type: detailed
🖼️  Source: https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcRPQ_gGhRNQdk5gxBqyA9diGOVps2ZliOpWX2ifjfPSPg&s...


{'detailed': 'Carte politique de la France métropolitaine. Fond rose saumon; '
             'mers en bleu (Manche au nord‑ouest, Océan Atlantique à l’ouest, '
             'Méditerranée au sud). Régions et départements bordés d’orange; '
             'villes en points noirs (Paris au centre‑nord). Pays voisins '
             'nommés autour. Corse en encart en bas à droite. Fleuves bleus, '
             'relief pâle, échelle en bas.',
 'educational': 'Résumé pédagogique de l’image\n'
                '\n'
                '- Type de document: c’est une carte de la France. Elle montre '
                'les régions (ancienne carte avant 2016), les principales '
                'villes, les fleuves, les montagnes, les frontières et les '
                'mers.\n'
                '\n'
                '- Les mers et océans: au nord la Manche, à l’ouest l’océan '
                'Atlantique, au sud la mer Méditerranée. La Corse est une île '
                'en Méditerranée.\n'
               

## 🎯 Summary and Best Practices

### ✅ What you have learned

- [ ] **GPT-5 configuration** : OpenRouter API and optimal parameters
- [ ] **Multimodal analysis** : Combining text + image
- [ ] **Educational prompts** : Adapting to the teaching level
- [ ] **Pedagogical use cases** : Concrete applications by domain
- [ ] **Optimization** : Quality and performance parameters

### 🚀 Next steps

1. **Experiment** with your own educational images
2. **Test** different analysis levels depending on your audience
3. **Integrate** GPT-5 into your pedagogical workflows
4. **Combine** with DALL-E 3 for generation + analysis
5. **Explore** the advanced notebooks (Module 02)

### 💡 Usage tips

**✅ Best practices:**
- Use high-quality images
- Adapt the prompt to the educational level
- Combine visual and contextual analysis
- Generate pedagogical questions
- Save successful analyses

**❌ Avoid:**
- Images that are too small or of poor quality
- Vague or generic prompts
- Forgetting the educational context
- Ignoring the model's limitations
- Not checking factuality

### 🔗 Additional resources

- **OpenRouter documentation** : [openrouter.ai](https://openrouter.ai)
- **GPT-5 guide** : Multimodal capabilities
- **Educational templates** : `docs/genai-phase2-templates.md`
- **CoursIA standards** : `docs/genai-images-development-standards.md`

### 📖 Scholarly references

- **OpenAI** (2025). *GPT-5 System Card & Model Page*. — Multimodal model (text + vision), **400,000 token** context window, 128,000 maximum output tokens. Official sources: openai.com/gpt-5/ and developers.openai.com/api/docs/models/gpt-5.
- **OpenAI** (2024). *GPT-4o System Card*. — Multimodal precursor of GPT-5 (vision + text unified), basis of the multimodal API used via OpenRouter in this notebook.


### Reading the result — what the next cell really presents

The `print('Cas d'usage pédagogiques')` above is a **static catalog**, not an execution. The five domains listed are there to give the student a structured starting point — each `prompt_template` is immediately reusable via `analyze_image_with_gpt5(image, prompt=...)`, without having to rewrite it. The `print('Template de prompt disponible')` is the clue that the cell has already played its pedagogical role by the time the student reads it, even if nothing was executed.

**Three things to remember for the summary cell** :
1. The OpenRouter API exposes a large family of GPT-5 models (the execution inventory is in the connection output) — the nomenclature is wider than the official OpenAI nomenclature, and the selection is made on the `contexte × coût × capabilities` trade-off (vision, tools, etc.).
2. The three analysis modes (quick/detailed/educational) are not interchangeable: quick is optimized for WCAG alt-text, educational produces student-guiding questions — it is the only mode that provides a turnkey pedagogical deliverable.
3. The multimodal pipeline (encode → POST → parse) is uniform whatever the image — the complexity lies in the prompt, not in the transport.

***

## Guided examples: three measured gestures

Three guided examples from a student contribution close the « analysis » part of the notebook: measuring the sensitivity of an analysis to the prompt wording (lexical similarity between two texts at constant format), adapting an alt text to its publication context under a length limit verified by measurement, and comparing two images from the same domain with an identical analysis prompt. The code is kept as it was delivered — it runs and prints its own checks.

### Guided example: neutral prompt vs oriented prompt — measuring the sensitivity to wording

*Student contribution (PR #23305, github.com/Perr1neF0), integrated as a guided example.*

The same image analyzed by GPT-5 can produce very different answers depending on the prompt wording. The example measures this sensitivity instead of suffering it: two prompts differing only in their **angle** (neutral vs oriented toward the history of science), same image, same imposed answer format.

**Key points**:
- The answer format is factored into a single variable (`format_sortie`) — if the format also differed, the comparison would no longer measure the wording
- The comparison is **quantified**: a lexical Jaccard over words of five letters and more, computed by `mots_cles()`, replaces eyeball estimation
- A check verifies that the three requested sections are present in each text; without it, a gap could be a mere format violation
- The printed conclusion links each wording to a use: visual inventory on one side, disciplinary course on the other

In [15]:
# Exemple guide : prompt neutre vs prompt oriente -- mesurer la sensibilite de la formulation
# (contribution etudiante, PR #23305 -- github.com/Perr1neF0)
import re

# Etape 1 : Choix de l'image
image_exo1 = example_images[1]['url']  # Laboratoire (Science et Technologie)

# Etape 2 : Redaction des deux prompts
format_sortie = f"""Réponds en {language}, avec exactement ces 3 sections :
**1. Éléments visibles** : liste à puces (5 à 8 éléments)
**2. Analyse** : 5 lignes maximum
**3. Conclusion** : une seule phrase"""
prompt_neutre = (
    "Décris cette image de façon objective, sans cadre théorique particulier.\n\n"
    + format_sortie
)
prompt_oriente = (
    "Analyse cette image sous l'angle de l'histoire des sciences : "
    "instruments, méthodes de travail et évolution des pratiques de laboratoire.\n\n"
    + format_sortie
)
# Indice : seul l'angle change -- le format de sortie reste identique

# Etape 3 : Lancement des deux analyses
resultat_neutre = analyze_image_with_gpt5(image_exo1, prompt=prompt_neutre)
resultat_oriente = analyze_image_with_gpt5(image_exo1, prompt=prompt_oriente)

# Etape 4 : Comparaison mesuree
def mots_cles(texte, taille_min=5):
    """Ensemble des mots significatifs (hors mots courts) d'un texte."""
    return {m for m in re.findall(r"[a-zàâçéèêëîïôûùüÿœ'-]+", texte.lower())
            if len(m) >= taille_min}

if resultat_neutre['success'] and resultat_oriente['success']:
    t_neutre = resultat_neutre['analysis']
    t_oriente = resultat_oriente['analysis']
    m_neutre, m_oriente = mots_cles(t_neutre), mots_cles(t_oriente)

    print(f"Longueur : neutre={len(t_neutre)} car. | orienté={len(t_oriente)} car.")
    print(f"Mots communs : {len(m_neutre & m_oriente)}")
    print(f"Jaccard (similarité lexicale) : "
          f"{len(m_neutre & m_oriente) / len(m_neutre | m_oriente):.2f}")
    print("\nSeulement dans le NEUTRE  :", sorted(m_neutre - m_oriente)[:15])
    print("Seulement dans l'ORIENTÉ  :", sorted(m_oriente - m_neutre)[:15])

    # Sections respectees ? (le format doit etre identique pour que la comparaison soit valide)
    for nom, t in [("neutre", t_neutre), ("orienté", t_oriente)]:
        ok = all(s in t.lower() for s in ["éléments visibles", "analyse", "conclusion"])
        print(f"Format respecté ({nom}) : {'OUI' if ok else 'NON'}")

    print("\n--- NEUTRE ---\n", t_neutre)
    print("\n--- ORIENTÉ ---\n", t_oriente)

    print("\nConclusion : le prompt neutre convient à un inventaire visuel (observation, "
          "alt text, premier contact avec l'image) ; le prompt orienté convient à un "
          "cours disciplinaire, car il fait émerger interprétation et vocabulaire du cadre choisi.")
else:
    for nom, r in [("neutre", resultat_neutre), ("orienté", resultat_oriente)]:
        if not r['success']:
            print(f"Échec ({nom}) : {r['error']}")


🔍 Analyse en cours...
📝 Type: detailed
🖼️  Source: https://images.unsplash.com/photo-1532094349884-543bc11b234d?w=800



🔍 Analyse en cours...
📝 Type: detailed
🖼️  Source: https://images.unsplash.com/photo-1532094349884-543bc11b234d?w=800


Longueur : neutre=1065 car. | orienté=1351 car.
Mots communs : 34
Jaccard (similarité lexicale) : 0.24

Seulement dans le NEUTRE  : ['accent', 'atmosphère', 'cadrage', 'champ', 'clarté', 'communique', 'contrôlée', 'couleur', 'couleurs', 'cours', 'disposition', 'donnant', 'ensemble', 'environnement', 'faible']
Seulement dans l'ORIENTÉ  : ['ambiance', 'après', 'artisanale', 'aseptisé', 'automatisés', 'biomédecine', 'borosilicate', 'borosilicaté', 'cette', 'chimie', 'classiques', 'coexistent', 'comme', 'condense', 'consommables']
Format respecté (neutre) : OUI
Format respecté (orienté) : OUI

--- NEUTRE ---
 **1. Éléments visibles**
- Plusieurs béchers en verre transparents de tailles différentes
- Une pipette compte-gouttes insérée dans un bécher contenant un liquide clair légèrement teinté
- Un flacon conique (type Erlenmeyer) vide en arrière-plan
- Un plan de travail lisse et clair, de couleur blanche
- Des reflets et transparences marqués sur les surfaces en verre
- Un arrière-plan fl

### Guided example: contextual alt text — one image, three contexts

*Student contribution (PR #23305, github.com/Perr1neF0), integrated as a guided example.*

The accessibility guided example showed three **lengths** of description for the same image. Here, it is the **publication context** that changes at constant length: a relevant alt text depends on what the reader came for.

**Key points**:
- The three context instructions live in a `contextes` dictionary, the length constraint is shared (`LIMITE = 125`)
- The constraint is **verified by measurement** (`len(texte)`), not by eye, with a retry that hardens the instruction when the text exceeds the limit
- Two final checks: each text fits under the limit, and the three texts are **distinct** — without the second, three interchangeable rephrasings would pass the test

In [16]:
# Exemple guide : alt text contextuel -- une image, trois contextes
# (contribution etudiante, PR #23305 -- github.com/Perr1neF0)

# Etape 1 : Choix de l'image
image_exo2 = example_images[3]['url']  # Paysage naturel

# Etape 2 : Definition des trois contextes de publication
contextes = {
    "article de vulgarisation scientifique": (
        "Contexte : article de vulgarisation scientifique pour un public curieux "
        "mais non spécialiste. Privilégie le phénomène naturel ou géologique visible "
        "(relief, eau, végétation, climat)."
    ),
    "publication sur un reseau social": (
        "Contexte : publication sur un réseau social. Privilégie l'ambiance, la "
        "lumière et l'impression d'ensemble, avec un ton vivant mais factuel."
    ),
    "manuel scolaire de geographie": (
        "Contexte : manuel scolaire de géographie pour des élèves. Privilégie le "
        "vocabulaire géographique précis (type de paysage, composantes, échelle)."
    ),
}

# Etape 3 : Redaction d'un prompt par contexte
LIMITE = 125

def prompt_alt(consigne_contexte, limite=LIMITE):
    return (
        f"Génère un alt text en {language} pour cette image.\n"
        f"{consigne_contexte}\n"
        f"Contraintes : {limite} caractères MAXIMUM (espaces compris), "
        "description factuelle, une seule phrase, aucune interprétation subjective.\n"
        "Réponds UNIQUEMENT avec le texte, sans guillemets ni préambule."
    )

def generer_alt_text(consigne_contexte, tentatives=3):
    """Appelle GPT-5 et relance si la limite de 125 caracteres est depassee."""
    prompt = prompt_alt(consigne_contexte)
    texte = ""
    for _ in range(tentatives):
        r = analyze_image_with_gpt5(image_exo2, prompt=prompt)
        if not r['success']:
            return f"ERREUR : {r['error']}"
        texte = r['analysis'].strip().strip('"\'«» ')
        if len(texte) <= LIMITE:
            return texte
        # On renforce la contrainte avec la longueur reelle constatee
        prompt = prompt_alt(consigne_contexte, limite=LIMITE - 20) + (
            f"\nTa version précédente faisait {len(texte)} caractères : raccourcis-la."
        )
    return texte  # derniere version, meme si trop longue (signalee plus bas)

# Etape 4 : Generation et verification
alt_texts = {ctx: generer_alt_text(consigne) for ctx, consigne in contextes.items()}

print("--- Alt texts générés ---")
for ctx, texte in alt_texts.items():
    statut = "OK" if len(texte) <= LIMITE else "TROP LONG"
    print(f"[{statut}] ({len(texte):3d} car.) {ctx}\n    -> {texte}\n")

# Les trois textes doivent etre distincts
textes = [t.lower() for t in alt_texts.values()]
print("Trois textes distincts :", "OUI" if len(set(textes)) == len(textes) else "NON")
print("Tous <= 125 caractères  :", "OUI" if all(len(t) <= LIMITE for t in alt_texts.values()) else "NON")


🔍 Analyse en cours...
📝 Type: detailed
🖼️  Source: https://images.unsplash.com/photo-1506905925346-21bda4d32df4?w=800



🔍 Analyse en cours...
📝 Type: detailed
🖼️  Source: https://images.unsplash.com/photo-1506905925346-21bda4d32df4?w=800



🔍 Analyse en cours...
📝 Type: detailed
🖼️  Source: https://images.unsplash.com/photo-1506905925346-21bda4d32df4?w=800


--- Alt texts générés ---
[OK] (107 car.) article de vulgarisation scientifique
    -> Chaîne de montagnes enneigées surplombant une mer de nuages dans une vallée, sous un ciel rosé à l’horizon.

[OK] ( 89 car.) publication sur un reseau social
    -> Sommets enneigés surplombant une mer de nuages, ciel rosé et lumière rasante à l’horizon.

[OK] (108 car.) manuel scolaire de geographie
    -> Massif alpin: pics enneigés et arêtes rocheuses dominant une vallée glaciaire comblée par une mer de nuages.

Trois textes distincts : OUI
Tous <= 125 caractères  : OUI


### Guided example: image comparator — two images from the same domain side by side

*Student contribution (PR #23305, github.com/Perr1neF0), integrated as a guided example.*

Comparing two images from the same domain in a single workflow: individual analysis of each image, then cross-comparison of the two analyses.

**Key points**:
- The analysis prompt is **the same** for both images — this is the condition that makes the comparison mechanical rather than arbitrary
- The cross-comparison is a third call that receives both analyses and **joins image B**, so that the model checks its claims against the visible image
- A failed analysis interrupts the cross-comparison (immediate `return`): crossing a missing analysis would produce an invented verdict
- The total time is measured and returned with the result

In [17]:
# Exemple guide : comparateur d'images -- deux images du meme domaine cote a cote
# (contribution etudiante, PR #23305 -- github.com/Perr1neF0)
import pprint, time

# 1. Selection des deux images
domaine = "Géographie"
image_a = example_images[3]['url']  # Paysage naturel (Géographie/Environnement)
image_b = "https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcRPQ_gGhRNQdk5gxBqyA9diGOVps2ZliOpWX2ifjfPSPg&s=10"  # carte de France (image deja utilisee plus haut)

# 2. Prompt d'analyse pour chaque image
prompt_analyse = f"""Tu es professeur de {domaine}. Réponds en {language}.
Analyse cette image avec exactement cette structure :
**Type d'image** : (photo, carte, schéma...) en une ligne
**Éléments principaux** : 3 à 5 puces
**Information géographique** : 3 lignes maximum
**Intérêt pédagogique** : une phrase"""

# 3. Fonction de comparaison
def compare_images(url_a, url_b, prompt):
    """
    Analyse deux images du meme domaine et produit une comparaison structuree.

    Returns:
        Dict avec les deux analyses, leur comparaison et le temps total.
    """
    debut = time.time()
    comparaison = {}

    # a) Analyse individuelle, meme prompt pour les deux images
    for cle, url in [("analyse_a", url_a), ("analyse_b", url_b)]:
        r = analyze_image_with_gpt5(url, prompt=prompt)
        if not r['success']:
            comparaison[cle] = None
            comparaison["erreur"] = f"{cle} : {r['error']}"
            return comparaison  # inutile de croiser si une analyse manque
        comparaison[cle] = r['analysis']

    # b) Croisement : on injecte les deux analyses dans le prompt, et on joint
    #    l'image B pour que GPT-5 puisse verifier les affirmations sur l'image visible
    prompt_croisement = f"""Tu compares deux images de {domaine}. Réponds en {language}.
Voici l'analyse de l'image A :
\"\"\"{comparaison['analyse_a']}\"\"\"

Voici l'analyse de l'image B (l'image jointe) :
\"\"\"{comparaison['analyse_b']}\"\"\"

Produis exactement ces trois sections :
**Points communs** : 3 puces
**Différences** : 3 puces
**Verdict** : laquelle des deux images est la plus utile pour introduire le domaine à des étudiants, et pourquoi (3 lignes maximum)."""

    r = analyze_image_with_gpt5(url_b, prompt=prompt_croisement)
    comparaison["comparaison"] = r['analysis'] if r['success'] else f"ERREUR : {r['error']}"
    comparaison["temps_total_s"] = round(time.time() - debut, 1)
    return comparaison

# 4. Executer et afficher
resultat_comparaison = compare_images(image_a, image_b, prompt_analyse)
pprint.pprint(resultat_comparaison, width=100)


🔍 Analyse en cours...
📝 Type: detailed
🖼️  Source: https://images.unsplash.com/photo-1506905925346-21bda4d32df4?w=800



🔍 Analyse en cours...
📝 Type: detailed
🖼️  Source: https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcRPQ_gGhRNQdk5gxBqyA9diGOVps2ZliOpWX2ifjfPSPg&s...



🔍 Analyse en cours...
📝 Type: detailed
🖼️  Source: https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcRPQ_gGhRNQdk5gxBqyA9diGOVps2ZliOpWX2ifjfPSPg&s...


{'analyse_a': "**Type d'image** : Photo de paysage de haute montagne au lever/au coucher du "
              'soleil\n'
              '\n'
              '**Éléments principaux** :\n'
              '- Chaîne de sommets enneigés aux arêtes aiguës\n'
              '- Mer de nuages comblant une large vallée\n'
              '- Versants rocheux sombres, traces de glaciers\n'
              '- Ciel rosé-orangé de lumière rasante\n'
              '\n'
              '**Information géographique** :\n'
              'Relief alpin typique, vallées en auge façonnées par l’érosion glaciaire.  \n'
              'Inversion thermique: air froid et humide piégé en fond de vallée formant une « mer '
              'de nuages ».  \n'
              'Fort gradient altitudinal avec étagement des milieux (neige/glace en altitude, '
              'vallée sous nuages).\n'
              '\n'
              '**Intérêt pédagogique** : L’image permet d’expliquer l’articulation relief–climat '
              'en montagn

## Exercises to complete

The three exercises that follow address what these examples do not yet measure: the **stability** of a repeated response at constant prompt, the **fidelity** of an alt text to the elements actually visible, and the **structuring** of an evaluation score into validatable JSON. Each skeleton runs without error as is — it is up to you to fill in the `TODO etudiant`.

### Exercise 1 : Reproducibility — the same question asked three times

The guided example *neutral prompt vs oriented prompt* measured the effect of the prompt's **angle**. Here, everything is frozen: same image, same prompt, same model. The question becomes *is the answer stable?* An inference model samples, and two identical calls may not return the same text.

**Goal**: measure the dispersion of three identical analyses, then examine what the API **advertises** as a stabilization lever.

**A fact to know before step 4** — verified on the API: `openai/gpt-5` lists **neither `temperature` nor `top_p`** among its supported parameters (`supported_parameters` field of `GET /models/openai/gpt-5/endpoints`: `reasoning`, `seed`, `max_tokens`…). This establishes an **absence of advertised support**, not a behavior: the metadata does not say what the provider does with a `temperature` value sent anyway (rejection, routing, normalization) — that actual behavior is verified in the API response, not in the metadata. The **advertised** stabilization parameter is `seed`.

**Steps**:
1. Choose an image and a short prompt with fixed sections
2. Run **three** identical analyses and keep the three texts
3. Measure the pairwise similarity — a lexical Jaccard as in the guided example (reuse `mots_cles`), or a simple rate of common sections — and present the result as **descriptive**
4. Consult the model's `supported_parameters` and conclude on the **advertised** support for `temperature`, without inferring a behavior
5. Conclude in one sentence: which lever does the API advertise to bring two runs closer, and for which use is a stable answer needed?

**Optional — two paid calls on a non-advertised parameter**: the skeleton suggests in step 4 reassigning `temperature = 0.0` then running again. The experiment is feasible but **not conclusive as it stands**: comparing the dispersion of three samples with that of two samples proves neither honoring nor non-honoring of the parameter — it is a descriptive observation, to be presented as such. If you do it, restore `temperature = 0.7` at the end of the cell.

**Extension — the advertised lever**: `seed` is not exposed by `analyze_image_with_gpt5()`. Using it requires **explicitly adapting the function** (adding `"seed": ...` to the request payload), then measuring whether two runs with a fixed `seed` actually get closer.

**Hints**:
- `analyze_image_with_gpt5()` reads `temperature` from the notebook's global variables: a reassignment in a cell is enough, without modifying the function
- supported parameters of a model: `GET https://openrouter.ai/api/v1/models/openai/gpt-5/endpoints`, field `supported_parameters`
- three identical calls cost three times the price of one call — keep the prompt short (`analysis_type="quick"` or few sections)
- a Jaccard close to 1 signals two nearly identical texts, a low Jaccard a complete rephrasing

In [18]:
# Exercice 1 : Reproductibilite -- la meme question posee trois fois
# TODO etudiant : mesurer la dispersion de trois analyses identiques, puis a temperature = 0.0

# Etape 1 : Image et prompt fixes (aucune variation d'angle ici)
image_exo1_rep = example_images[3]['url']
prompt_repete = f"""Réponds en {language}, avec exactement ces 2 sections :
**1. Éléments visibles** : 3 puces
**2. Conclusion** : une seule phrase"""

# Etape 2 : Trois analyses identiques
# TODO etudiant : construire une liste de trois analyses via
#                 analyze_image_with_gpt5(image_exo1_rep, prompt=prompt_repete)
analyses_repetees = []  # TODO etudiant

# Etape 3 : Similarite deux a deux
# TODO etudiant : calculer la similarite de chaque paire (Jaccard lexical via mots_cles,
#                 ou taux de sections communes) et afficher les valeurs
similarites = []  # TODO etudiant

# Etape 4 : Effet de la temperature
# TODO etudiant : reassigner temperature = 0.0, relancer deux analyses, comparer la dispersion,
#                 puis restaurer temperature = 0.7
analyses_froides = []  # TODO etudiant

# Etape 5 : Conclusion en une phrase (a afficher)
print("Exercice a completer")

Exercice a completer


### Exercise 2 : Fidelity audit — an alt text claims only what is visible

The guided example *contextual alt text* produced three texts compliant with a length limit and a context. One quality escapes these two controls: **fidelity**. A short, well-turned alt text can cite a color, an object or a place the image does not contain — and the screen reader cannot contradict it.

**Goal**: build an audit in two steps — an inventory of the visible elements, then an alt text checked term by term against that inventory.

**Steps**:
1. Choose an image among `example_images`
2. Ask for an **inventory**: bulleted list of the visible elements, without interpretation
3. Ask for an **alt text** of 125 characters maximum for the same image
4. Extract the significant terms of the alt text (reuse `mots_cles`) and note those that do not appear in the inventory
5. Display the verdict and recall the limit of the method

**Why two separate calls?** Calls to `analyze_image_with_gpt5()` are **independent**: no conversational history circulates from one to the other, and the order of the two requests therefore has no causal effect. The point of separating them lies elsewhere: two short, targeted prompts, instead of a merged prompt where the model would see the alt text while drawing up the inventory — and would answer its own conclusion accordingly.

**Hints**:
- a term absent from the inventory is not necessarily invented (synonym, rephrasing): the verdict says « to revise », not « false »
- the inventory and the alt text are two **distinct measurements** of the same image: comparing them only makes sense if each was produced under a single instruction

In [19]:
# Exercice 2 : Audit de fidelite -- un alt text n'affirme que ce qui est visible
# TODO etudiant : inventaire des elements visibles, alt text, puis verification terme a terme

# Etape 1 : Image
image_exo2_audit = example_images[2]['url']

# Etape 2 : Inventaire des elements visibles (demande AVANT l'alt text)
prompt_inventaire = f"""Liste en {language} les éléments visibles de cette image.
Réponds uniquement par une liste à puces de 5 à 8 éléments factuels, sans interprétation."""
# TODO etudiant : inventaire = analyze_image_with_gpt5(image_exo2_audit, prompt=prompt_inventaire)
inventaire = None  # TODO etudiant

# Etape 3 : Alt text soumis a l'audit
prompt_alt_audit = f"""Génère un alt text en {language} de 125 caractères maximum,
une seule phrase factuelle, sans interprétation subjective."""
# TODO etudiant : alt_text_audit = analyze_image_with_gpt5(image_exo2_audit, prompt=prompt_alt_audit)
alt_text_audit = None  # TODO etudiant

# Etape 4 : Verification terme a terme contre l'inventaire
# TODO etudiant : termes de l'alt text (mots_cles) absents de l'inventaire (mots_cles)
termes_non_etayes = []  # TODO etudiant

# Etape 5 : Verdict
# TODO etudiant : afficher la liste des termes non etayes et rappeler la limite de la
#                 verification lexicale (un synonyme n'est pas une invention)
print("Exercice a completer")

Exercice a completer


### Exercise 3 : Structured evaluation — grading a response into validatable JSON

The guided examples produce free-text analyses. A common use in teaching is the opposite: **grade** a response against explicit criteria, and make that score usable by a program. The difficulty is not the score itself, it is obtaining a **structured and validatable** response — a model that returns a sentence of appreciation does not return a score.

**Goal**: write a function that grades an alt text against three criteria and returns a strict JSON, then **validate** that JSON (parsing, keys, bounds).

**Steps**:
1. Choose the image and the response to grade (an alt text generated above will do)
2. Define a rubric of three criteria, each scored from 0 to 2
3. Write a prompt that requires **only** a JSON object with the expected keys — no preamble, no markdown code block
4. Parse the response (`json.loads`) and validate: three criteria present, each score an integer between 0 and 2
5. Display the total score, and **report** the cases where parsing fails instead of working around them

**Hints**:
- `json.loads()` fails on a markdown code block: ask for the raw text, without tags
- strict validation (`isinstance(..., int)` and bounds) turns « looks fine » into a binary verdict
- the grader is a **stochastic** model: the score is indicative, not a truth — its own repeatability can be measured (run the evaluation again identically and compare the two scores)

In [20]:
# Exercice 3 : Evaluation structuree -- noter un alt text avec une rubrique, en JSON
# TODO etudiant : obtenir une note JSON valide (3 criteres, chacun de 0 a 2)

# Etape 1 : La reponse a evaluer
reponse_a_noter = "Vue aérienne d'une ville au crépuscule avec un dôme doré et des montagnes."  # TODO etudiant : votre propre alt text
image_eval = example_images[0]['url']

# Etape 2 : La rubrique (3 criteres, notes de 0 a 2)
RUBRIQUE = {
    "fidelite": "0-2 : chaque element de l'alt text est visible sur l'image",
    "longueur": "0-2 : respecte la limite de 125 caracteres",
    "objectivite": "0-2 : aucune interpretation subjective",
}

# Etape 3 : Prompt exigeant un JSON strict
# TODO etudiant : rediger le prompt (rubrique + reponse + format JSON exact, sans preambule)

# Etape 4 : Appel, parsing et validation
# TODO etudiant : note = json.loads(reponse) puis verifier les cles, le type entier et les bornes
note_json = None  # TODO etudiant

# Etape 5 : Affichage du verdict
print("Exercice a completer")

Exercice a completer


### Success criteria

**Exercise 1 — reproducibility**
- [ ] Three identical analyses run, and their pairwise similarities measured (not estimated), presented as **descriptive**
- [ ] The model's `supported_parameters` consulted, the conclusion limited to the **advertised** support of `temperature` — no behavior (ignoring, rejection, normalization) inferred from the metadata alone
- [ ] Any comparison of additional samples (optional, paid) remains descriptive, never presented as causal proof; `temperature` is restored to `0.7` at the end of the cell if the experiment is done

**Exercise 2 — fidelity audit**
- [ ] Inventory and alt text produced by two **separate** calls, each under a single instruction
- [ ] The terms of the alt text are checked against the inventory by a computation, with the list of unsupported terms
- [ ] The verdict says « to revise » and names the limit of the lexical verification

**Exercise 3 — structured evaluation**
- [ ] The rubric has three criteria scored from 0 to 2
- [ ] The JSON is parsed and validated (keys present, integers, bounds) — a parsing failure is reported, never worked around
- [ ] The total score and the failure cases are displayed, with the **indicative** nature of the score recalled

### Extension (optional)

For advanced students:
- Expose `seed` in `analyze_image_with_gpt5()` (explicit payload adaptation) and measure whether two runs with a fixed `seed` get closer
- Create an **interactive quiz** based on the generated questions
- Extend the comparator to **three images** or more (comparison matrix)
- Replay exercise 3 with a second independent evaluation (same rubric, different prompt) and measure the agreement between the two scores

### Reading the result — the exercises in synthesis

The three guided examples of the previous section each show a measured gesture: comparing two wordings at constant format, adapting an alt text to its context under a length constraint, crossing two analyses with an identical prompt. The three exercises that complete them address what these gestures do not yet measure: the **stability** of a repeated response, the **fidelity** of an alt text to the visible elements, and the **structuring** of a score into validatable JSON. The notebook runs end to end as long as the exercises remain stubs.

## Conclusion

GPT-5 image generation (via OpenRouter) embodies the **native multimodal** approach: the model does not merely generate images from a text prompt, it analyzes and produces in a single flow, which opens three distinct pedagogical usage modes — quick, detailed, educational — richness and cost grow together (the measurements live in the table printed by the test). The inference cost / content richness ratio is directly exposed to the user by the choice of mode.

**Specific contribution of native multimodality**: the automatic generation of alt text (a single sentence under the 125-character threshold of the WCAG accessibility referentials) integrates image description as a **first-class deliverable**, not as an optional post-processing. For the 5 educational use cases covered (History, Science, Geography, Art and Culture, Medicine), this is a substantial accessibility gain over purely text-to-image models.

**Positioning in the ecosystem**: GPT-5 completes the course's palette by covering the *comprehension + generation* axis, where DALL-E 3 focuses on generation alone and SD XL Turbo exposes the internal workings. The large family of GPT-5 models exposed by the OpenRouter API opens a wide comparative experimentation ground but introduces a strong dependency on the routing provider.

**To go further**: explore the `04-1-Educational-Content-Generation` notebook which orchestrates several models to produce a complete pedagogical deliverable (image + alt text + structured metadata).